<a href="https://colab.research.google.com/github/EpiPandit/MPM_200/blob/main/Session%205/Introduction%20to%20R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐄 R Quest: Your First Day as a Data Detective
**MPM 200 · Week 1 · Introduction to R** · UC Davis School of Veterinary Medicine

> **The story.** It's August 2020. You've just joined a state public-health team as the new epidemiologist. Your supervisor drops a file on your desk (five months of daily COVID-19 reports from every US state and territory) and says: *"Tell me what's in here by Friday."*
>
> You've never used R. By the end of this notebook you will have loaded, cleaned, summarized and plotted that data, and drawn California's epidemic curve.

**How the quest works**

| | |
|---|---|
| 🧭 **6 levels + a boss level** | Each one adds a single new skill. Run the cells in order. |
| 🔮 **Predict first** | Before some cells, jot down a guess. Being wrong is where the learning happens. |
| 🎯 **Your turn** | Short challenges. These cells have blanks (`____`), so they won't run until you fill them in. |
| 💡 **Hints** | Hidden in toggles. Try for two minutes before you peek. |
| 🐛 **Bug hunt** | Errors are normal. Reading them is a skill, and we'll practise it. |

## 🚀 Level 0 · Set up Colab (2 minutes)

1. Open this notebook with the **Open in Colab** badge and sign in with your **UC Davis** Google account.
2. **Runtime → Change runtime type → R**. Leave hardware on CPU.
3. **File → Save a copy in Drive** so the edits are yours (copies land in *My Drive / Colab Notebooks*).
4. Run a cell with **Shift + Enter**.
5. Colab forgets everything (installed packages, uploaded files, objects) when the runtime disconnects. When you come back, run the cells from the top again.

If the next cell prints an R version, you're in. 🎉

In [ ]:
R.version.string

## 🧮 Level 1 · R is a very fancy calculator

R evaluates whatever you type. To keep a result, store it in an **object** with `<-` (you'll also see `=`; both work).
Anything after `#` is a comment: R ignores it, humans love it.

In [ ]:
2 + 2
12 * 7        # dogs in the ward × doses per day
sqrt(144)

In [ ]:
n_cats <- 14          # an object holding a number
n_dogs <- 9
n_cats + n_dogs

### What kind of thing is it? Data types
R treats text, numbers and TRUE/FALSE differently. `class()` tells you what you have.

In [ ]:
patient    <- "Biscuit"   # character (text) needs quotes
weight_kg  <- 4.2         # numeric
vaccinated <- TRUE        # logical: TRUE or FALSE

class(patient)
class(weight_kg)
class(vaccinated)

### Vectors: many values in one object
`c()` **c**ombines values. Most R functions work on a whole vector at once, with no loops needed.

In [ ]:
ward_weights <- c(4.2, 5.1, 3.8, 6.0, 4.9)   # five cats, in kg

mean(ward_weights)
max(ward_weights)
ward_weights * 2.2      # kg → lb, all five at once
ward_weights > 5        # which cats weigh more than 5 kg?

### 🎯 Your turn
Create a vector called `species` with the last **six** patients seen at the VMTH (repeats are fine, e.g. `"dog", "cat", "dog", ...`).
1. How many patients? Use `length()`.
2. How many of each species? Use `table()`.

<details><summary>💡 <b>Hint</b> (try first!)</summary>

`species <- c("dog", "cat", "dog", "horse", "cat", "dog")` then `length(species)` and `table(species)`

</details>

In [ ]:
species <- c(____)
length(____)
table(____)

## 📦 Level 2 · Packages: R's app store

A **package** is a bundle of functions somebody else wrote and shared. Using one takes two steps:

| Step | Command | How often |
|---|---|---|
| **Install** (download it) | `install.packages("name")` | Once per laptop. **Once per session on Colab** (it forgets). |
| **Load** (switch it on) | `library(name)` | Every session |

Let's install a very serious scientific package.

In [ ]:
install.packages("cowsay")

Every function has a help page. `?function_name` opens it. Read the **Arguments** and **Examples** sections, and get into the habit of doing this for every new function you meet.

In [ ]:
?install.packages

In [ ]:
library(cowsay)
say("Hello, MPVM class! Welcome to R.", by = "cow")

In [ ]:
?say
names(animals)   # every animal cowsay knows

### 🎯 Your turn
1. Make the **bat** say something nice about bats (they eat a *lot* of insects).
2. Pick your favourite animal from `names(animals)` and have it introduce **you**.
3. Bonus: try the `what_color` and `by_color` arguments (e.g. `"blue"`, `"yellow"`). Colors may not show in every viewer; that's fine.

<details><summary>💡 <b>Hint</b> (try first!)</summary>

`say("Bats are pest control with wings!", by = "bat", what_color = "blue", by_color = "red")`

</details>

In [ ]:
say("____", by = "bat")

## 🗂️ Level 3 · Know where you are: the working directory

R reads and writes files relative to a **working directory**. On Colab it is `/content`. Click the 📁 folder icon on the left to see it.

In [ ]:
getwd()

In [ ]:
list.files()     # what is in here right now?

Remember the lecture: **protect raw data and keep outputs separate.** Let's build a tiny, tidy project.

In [ ]:
dir.create("data_raw", showWarnings = FALSE)
dir.create("outputs",  showWarnings = FALSE)
list.files()

> **About `setwd()`.** It changes the working directory. On your laptop it might be `setwd("C:/Users/you/MPM200")` (Windows) or `setwd("~/MPM200")` (Mac).
> A path from *someone else's* computer will always fail on yours, so copy-pasting `setwd("C:/Users/prana/...")` won't work. On Colab we stay in `/content` and use **relative paths** like `"outputs/plot.pdf"`.

### Your environment: which objects exist?

In [ ]:
ls()

In [ ]:
rm(list = ls())   # deletes ALL objects. There is no undo!
ls()              # empty again

## 🔍 Level 4 · Meet the data

The **COVID Tracking Project** was a volunteer effort that compiled daily state-level reports during 2020–21. Our file is a snapshot up to **7 August 2020**: one row per state (or territory) per day.

We download it straight from the course GitHub into `data_raw/` (no upload needed), then read it.

In [ ]:
library(dplyr)
library(ggplot2)

url <- "https://raw.githubusercontent.com/EpiPandit/MPM_200/main/data/covid-tracking-data-master_COVID_Tracking_Project_2020_08_07_daily.csv"
download.file(url, destfile = "data_raw/covid_tracking_20200807.csv")

COVID <- read.csv("data_raw/covid_tracking_20200807.csv")

### 🔮 Predict first
One row per state per day, about 56 states and territories, about five months of reports.
**Roughly how many rows do you expect?** Write your guess here: `____`

In [ ]:
dim(COVID)      # rows, columns
nrow(COVID)
ncol(COVID)

In [ ]:
head(COVID)    # first 6 rows

In [ ]:
colnames(COVID)

In [ ]:
str(COVID)     # structure: the type of every column

### One column at a time: `$`
`data$column` pulls a single column out as a vector.

In [ ]:
head(COVID$state, 20)

In [ ]:
COVID$state <- as.factor(COVID$state)   # text → categories (factor)
levels(COVID$state)
length(unique(COVID$state))             # how many states + territories?

In [ ]:
table(COVID$state)    # rows per state

🤔 **Think:** almost every state has about 155 rows, but **WA has 199**. What does that tell you about *when* Washington started reporting? (Hint: where was the first US case detected?)

### Missing values (`NA`)
`is.na()` returns TRUE where a value is missing. TRUE counts as 1, so `sum()` counts the gaps.

In [ ]:
sum(is.na(COVID$state))                   # state is never missing
sum(is.na(COVID$hospitalizedCurrently))   # hospitalization often is

In [ ]:
colSums(is.na(COVID))   # missing values in every column at once

### 🎯 Your turn
Which **three** columns have the most missing data? Would you trust an analysis built on them?

<details><summary>💡 <b>Hint</b> (try first!)</summary>

`sort(colSums(is.na(COVID)), decreasing = TRUE)[1:3]`

</details>

In [ ]:
# your code here

### 🐛 Bug hunt: round brackets vs square brackets
Rows and columns are picked with **square** brackets: `data[row, column]`. **Round** brackets `()` *call functions*.

Remove the `#` in the next cell, run it, and read the error message. What is R telling you? Then fix it.

In [ ]:
# COVID(1, 2)    # 🐛 I want row 1, column 2

In [ ]:
COVID[1, 2]                                   # the fix
COVID[1:3, c("date", "state", "positive")]    # rows 1–3, three named columns

### Subsetting rows: California only

In [ ]:
CA <- COVID[COVID$state == "CA", ]
dim(CA)
head(CA[, c("date", "state", "positive", "positiveIncrease")])

## 🧰 Level 5 · Wrangling with dplyr

Four verbs do most of the work in data management:

| Verb | What it does |
|---|---|
| `select()` | keep some **columns** |
| `filter()` | keep some **rows** |
| `group_by()` | split the data into groups |
| `summarise()` | collapse each group into a summary |

The pipe `%>%` reads as **"and then"**. (RStudio shortcut: Ctrl/Cmd + Shift + M. Base R also has `|>`.)

In [ ]:
COVID %>%
  select(date, state, positive, positiveIncrease) %>%
  head()

In [ ]:
COVID %>%
  filter(state == "CA", date >= 20200801) %>%     # dates are stored as numbers like 20200807
  select(date, state, positiveIncrease)

### Total cases per state
Two columns look similar, but they mean different things:
* `positive`: the **cumulative** count (running total so far)
* `positiveIncrease`: **new** cases reported that day (incidence)

Summing the daily new cases gives the total. Let's check that against the last cumulative value.

🔮 **Predict first:** which **five** states had the most reported cases by 7 August 2020? `____, ____, ____, ____, ____`

In [ ]:
total_cases <- COVID %>%
  group_by(state) %>%
  summarise(total_cases     = sum(positiveIncrease, na.rm = TRUE),
            last_cumulative = max(positive, na.rm = TRUE)) %>%
  arrange(desc(total_cases))

head(total_cases, 5)

🤔 The two columns are *almost* equal, but not quite. Why might a running total disagree slightly with the sum of daily reports? (Think: corrections and back-filled data.)

### 🎯 Your turn
Which five states had reported the most **deaths**? Use the `deathIncrease` column.

<details><summary>💡 <b>Hint</b> (try first!)</summary>


```r
COVID %>%
  group_by(state) %>%
  summarise(total_deaths = sum(deathIncrease, na.rm = TRUE)) %>%
  arrange(desc(total_deaths)) %>%
  head(5)
```

</details>

In [ ]:
COVID %>%
  group_by(____) %>%
  summarise(total_deaths = sum(____, na.rm = TRUE)) %>%
  arrange(desc(____)) %>%
  head(5)

### Save your summary table (`row.names = FALSE` keeps R's row numbers out of the file)

In [ ]:
write.csv(total_cases, "outputs/total_cases_by_state.csv", row.names = FALSE)
list.files("outputs")

## 🎨 Level 6 · ggplot2: build a plot one layer at a time

ggplot2 follows the **grammar of graphics**: **data** + **aes**thetics (what goes on which axis) + **geom**etry (bars, points, lines) + extras. Layers are added with `+`.
Run these cells one at a time and watch the plot grow.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)   # plot size in Colab

In [ ]:
ggplot(total_cases)                                     # 1. an empty canvas

In [ ]:
ggplot(total_cases, aes(x = state, y = total_cases))    # 2. axes mapped

In [ ]:
ggplot(total_cases, aes(x = state, y = total_cases)) +
  geom_col()                                               # 3. bars

In [ ]:
ggplot(total_cases, aes(x = state, y = total_cases)) +
  geom_col() +
  coord_flip()                                             # 4. readable labels

In [ ]:
p <- ggplot(total_cases, aes(x = reorder(state, total_cases), y = total_cases)) +
  geom_col(fill = "#022851") +                             # 5. Aggie Blue, sorted
  coord_flip() +
  scale_y_continuous(labels = scales::comma) +
  labs(x = NULL,
       y = "Total reported cases (to 7 Aug 2020)",
       title = "COVID-19 cases by state",
       caption = "Source: The COVID Tracking Project") +
  theme_minimal(base_size = 11)
p

Prefer points to bars? Swap one layer: `geom_point(colour = "#022851", size = 2)`. Try it!

### Save the plot

In [ ]:
ggsave("outputs/cases_by_state.pdf", p, width = 6, height = 8)
list.files("outputs")

⬇️ **Download it:** 📁 → `outputs` → ⋮ next to the file → **Download**. Remember that Colab files vanish when the runtime disconnects.

### 🎯 Glow-up challenge
Make California stand out in **Aggie Gold** (`#FFBF00`) while every other state stays Aggie Blue.

<details><summary>💡 <b>Hint</b> (try first!)</summary>


Map the fill to a TRUE/FALSE question, then choose the two colours:
```r
ggplot(total_cases, aes(x = reorder(state, total_cases), y = total_cases,
                        fill = state == "CA")) +
  geom_col() +
  coord_flip() +
  scale_fill_manual(values = c("#022851", "#FFBF00"), guide = "none")
```

</details>

In [ ]:
# your glow-up here

## 👾 Boss level · Draw California's epidemic curve

An **epi curve** plots new cases over time. It is the single most-used chart in outbreak investigation.
Dates are stored as numbers like `20200807`, so first we turn them into real `Date` objects.

In [ ]:
CA <- COVID %>%
  filter(state == "CA") %>%
  mutate(date = as.Date(as.character(date), format = "%Y%m%d")) %>%
  arrange(date)

class(CA$date)
range(CA$date)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 4.5)

ggplot(CA, aes(x = date, y = positiveIncrease)) +
  geom_col(fill = "#022851") +
  labs(x = NULL, y = "New reported cases per day",
       title = "California: daily new COVID-19 cases, 2020") +
  theme_minimal()

### 🎯 Final challenges (pick at least one)
1. **Smooth it.** Daily reports are noisy (weekends!). Add a 7-day rolling average line in Aggie Gold.
2. **Go home.** Draw the epi curve for your home state, or any state you're curious about.
3. **Detective.** Some states have *negative* daily increases. Find them with `filter(COVID, positiveIncrease < 0)`. How can new cases be negative? What does that tell you about surveillance data?

<details><summary>💡 <b>Hint</b> (try first!)</summary>


Rolling average (note `stats::filter`, because dplyr has its own `filter()`):
```r
CA <- CA %>%
  mutate(avg7 = as.numeric(stats::filter(positiveIncrease, rep(1/7, 7), sides = 1)))

ggplot(CA, aes(x = date, y = positiveIncrease)) +
  geom_col(fill = "#022851") +
  geom_line(aes(y = avg7), colour = "#FFBF00", linewidth = 1.2) +
  theme_minimal()
```

</details>

In [ ]:
# boss-level code here

In [ ]:
say("Quest complete! You loaded, wrangled, plotted and saved real surveillance data.", by = "chicken")

## 🏁 Before you log off
- **Save your work:** File → Save a copy in Drive, *or* File → Save a copy in GitHub → **your own branch** (never `main`).
- **Download anything you need** from `outputs/`.
- **Cheat sheets worth bookmarking:** [dplyr](https://rstudio.github.io/cheatsheets/data-transformation.pdf) · [ggplot2](https://rstudio.github.io/cheatsheets/data-visualization.pdf)

Next up: the Week 1 assignment, **🐦 Will the Hummers Survive?**